# Perfume Recommender System
Using KNN with Leave-One-Out evaluation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from scipy.optimize import minimize

RNG = np.random.default_rng(42)

## 1. Load Data

In [ ]:
df = pd.read_json("data/perfumes.json")
df["ingredients"] = df["ingredients"].apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else [])
print(f"Total perfumes: {len(df)}")
df.sample(5)

## 2. Feature Engineering

In [ ]:
# --- Encode ingredients with MultiLabelBinarizer ---
mlb = MultiLabelBinarizer()
ingredients_encoded = mlb.fit_transform(df["ingredients"])
print(f"Unique ingredients: {len(mlb.classes_)}")

# --- Encode categorical features with OneHotEncoder ---
cat_cols = ["family", "subfamily", "gender"]
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_encoded = ohe.fit_transform(df[cat_cols])

# Split cat into named blocks for per-block weighting
n_family    = len(ohe.categories_[0])
n_subfamily = len(ohe.categories_[1])
n_gender    = len(ohe.categories_[2])
family_encoded    = cat_encoded[:, :n_family]
subfamily_encoded = cat_encoded[:, n_family:n_family + n_subfamily]
gender_encoded    = cat_encoded[:, n_family + n_subfamily:]
print(f"Blocks — ingredients:{ingredients_encoded.shape[1]}  "
      f"family:{n_family}  subfamily:{n_subfamily}  gender:{n_gender}")

# Unweighted baseline matrix (weights = [1,1,1,1])
def build_matrix(w):
    """Return feature matrix scaled by 4-element weight vector."""
    w_ing, w_fam, w_sub, w_gen = w
    return np.hstack([
        ingredients_encoded * w_ing,
        family_encoded      * w_fam,
        subfamily_encoded   * w_sub,
        gender_encoded      * w_gen,
    ]).astype(np.float32)

matrix = build_matrix([1, 1, 1, 1])
print(f"Feature matrix shape: {matrix.shape}")

## 3. Feature Engineering Module

In [ ]:
class fe:
    @staticmethod
    def build_query_vector(notes, family, subfamily, gender, mlb, ohe, w=None):
        """Build a weighted feature vector from partial perfume info.

        w : array-like of 4 weights [w_ing, w_fam, w_sub, w_gen].
            Defaults to [1, 1, 1, 1] (no weighting).
        """
        if w is None:
            w = [1, 1, 1, 1]
        w_ing, w_fam, w_sub, w_gen = w

        notes_vec = mlb.transform([notes])          # (1, n_ingredients)
        cat = pd.DataFrame([[family, subfamily, gender]],
                           columns=["family", "subfamily", "gender"])
        cat_enc = ohe.transform(cat)                # (1, n_cat)

        fam_vec = cat_enc[:, :n_family]
        sub_vec = cat_enc[:, n_family:n_family + n_subfamily]
        gen_vec = cat_enc[:, n_family + n_subfamily:]

        return np.hstack([
            notes_vec * w_ing,
            fam_vec   * w_fam,
            sub_vec   * w_sub,
            gen_vec   * w_gen,
        ]).astype(np.float32)

## 4. Leave-One-Out Evaluation

In [ ]:
def leave_one_out_hit_rate(df, mlb, ohe, matrix, model, sample=300, k=10, w=None):
    """Query with half of a perfume's notes; is the perfume back in top-K?

    Uses the provided fitted NearestNeighbors model.
    w : 4-element weight vector passed to fe.build_query_vector so the
        query is scaled the same way as the matrix.
    """
    if w is None:
        w = [1, 1, 1, 1]
    # Deterministic sample: a fresh seeded generator on every call so the LOO
    # hit-rate is repeatable for a given weight vector. Previously this used the
    # shared global RNG, so each call drew a DIFFERENT subset -> the objective
    # was noisy and the optimiser stalled at the starting point.
    rng = np.random.default_rng(42)
    idxs = rng.choice(len(df), size=min(sample, len(df)), replace=False)
    hits = 0
    evaluated = 0
    for i in idxs:
        row = df.iloc[i]
        notes = list(row["ingredients"])
        if len(notes) < 2:
            continue
        evaluated += 1
        half = rng.choice(notes, size=max(1, len(notes) // 2), replace=False).tolist()
        query = fe.build_query_vector(
            half, row["family"], row["subfamily"], row["gender"], mlb, ohe, w=w
        )
        _, indices = model.kneighbors(query)
        top = indices.ravel()[:k]
        if i in top:
            hits += 1
    return hits / evaluated if evaluated else 0.0

## 5. Hyperparameter Search

In [ ]:
metrics = ["euclidean", "manhattan", "cosine", "jaccard"]
n_values = [3, 5, 7, 10]

results = []

for metric in metrics:
    for n in n_values:
        model = NearestNeighbors(
            n_neighbors=n + 1,
            metric=metric,
            algorithm="brute"
        )
        model.fit(matrix)

        # Pass the fitted model so LOO uses the same metric being evaluated
        hit_rate = leave_one_out_hit_rate(df, mlb, ohe, matrix, model=model, sample=300, k=n)

        results.append({
            "metric": metric,
            "n_neighbors": n,
            "hit_rate": round(hit_rate, 4)
        })
        print(f"metric={metric:12s}  n={n:2d}  hit_rate={hit_rate:.4f}")

results_df = pd.DataFrame(results)
print("\nTop results:")
print(results_df.sort_values("hit_rate", ascending=False).head(10).to_string(index=False))

In [ ]:
# Pick the best config
best = results_df.sort_values("hit_rate", ascending=False).iloc[0]
print(f"Best config: metric={best.metric}, n_neighbors={int(best.n_neighbors)}, hit_rate={best.hit_rate}")

best_model = NearestNeighbors(
    n_neighbors=int(best.n_neighbors) + 1,
    metric=best.metric,
    algorithm="brute"
)
best_model.fit(matrix)

## 5b. Feature Weight Optimisation (scipy)

In [ ]:
# Fix the metric chosen from hyperparameter search for weight optimisation
EVAL_METRIC = best.metric          # e.g. 'euclidean'
EVAL_K      = int(best.n_neighbors)
OPT_SAMPLE  = 300                  # LOO samples per objective call

call_count = [0]

def objective(log_w):
    """Minimise negative LOO hit-rate.

    Optimise in log-space so weights stay positive without constraints.
    We fix w_ing=1 as the reference and learn the 3 relative weights,
    keeping the search space at 3 dims instead of 4.
    """
    call_count[0] += 1
    # w = [1, exp(a), exp(b), exp(c)]  — ingredients anchored at 1
    w = np.concatenate([[1.0], np.exp(log_w)])

    mat = build_matrix(w)
    model = NearestNeighbors(
        n_neighbors=EVAL_K + 1,
        metric=EVAL_METRIC,
        algorithm="brute",
    )
    model.fit(mat)

    hr = leave_one_out_hit_rate(
        df, mlb, ohe, mat, model,
        sample=OPT_SAMPLE, k=EVAL_K, w=w
    )
    print(f"  [{call_count[0]:3d}] w={np.round(w,3)}  hit_rate={hr:.4f}")
    return -hr

# Nelder-Mead: derivative-free. With x0 = zeros, SciPy's default simplex only
# perturbs each coordinate by 0.00025 (its rule for zero-valued coords), so the
# search never meaningfully left [1,1,1,1]. We supply an explicit initial simplex
# that explores up to ~2x relative weight changes, and tighten fatol now that the
# objective is deterministic.
x0 = np.zeros(3)   # log-space start = weights [1, 1, 1, 1]
_step = np.log(2.0)
initial_simplex = np.vstack([x0, x0 + np.eye(3) * _step])
print("Optimising feature weights (Nelder-Mead) …")
opt_result = minimize(
    objective,
    x0,
    method="Nelder-Mead",
    options=dict(maxiter=60, xatol=0.01, fatol=0.001, disp=True,
                 initial_simplex=initial_simplex),
)

best_w = np.concatenate([[1.0], np.exp(opt_result.x)])
print(f"\nOptimised weights  [w_ing, w_family, w_subfamily, w_gender] = {np.round(best_w, 3)}")
print(f"Best hit-rate  : {-opt_result.fun:.4f}")

## 6. Best Model & Recommend Function

In [ ]:
# Pick the best metric/k config
best = results_df.sort_values("hit_rate", ascending=False).iloc[0]
print(f"Best config: metric={best.metric}, n_neighbors={int(best.n_neighbors)}, hit_rate={best.hit_rate}")

best_model = NearestNeighbors(
    n_neighbors=int(best.n_neighbors) + 1,
    metric=best.metric,
    algorithm="brute"
)
best_model.fit(matrix)

In [ ]:
# Rebuild matrix and model with optimised weights
opt_matrix = build_matrix(best_w)
opt_model  = NearestNeighbors(
    n_neighbors=int(best.n_neighbors) + 1,
    metric=best.metric,
    algorithm="brute",
)
opt_model.fit(opt_matrix)
print("opt_model ready")


def recommend(notes, family, subfamily, gender,
              df=df, mlb=mlb, ohe=ohe,
              model=opt_model, w=best_w, k=5):
    """
    Recommend perfumes based on preferred notes and profile.

    Parameters
    ----------
    notes      : list of str  e.g. ['Jasmine', 'Rose', 'Bergamot']
    family     : str          e.g. 'FLORAL'
    subfamily  : str          e.g. 'AMBERY (ORIENTAL)'
    gender     : str          e.g. 'Female'
    model      : fitted NearestNeighbors  defaults to opt_model (weight-optimised)
    w          : 4-element weight vector  defaults to best_w from optimisation
    k          : int          number of recommendations
    """
    query = fe.build_query_vector(notes, family, subfamily, gender, mlb, ohe, w=w)
    distances, indices = model.kneighbors(query)
    top_idx = indices.ravel()[:k]
    recs = df.iloc[top_idx][["brand", "name_perfume", "family",
                              "subfamily", "gender", "ingredients"]].copy()
    recs["distance"] = distances.ravel()[:k].round(4)
    return recs.reset_index(drop=True)


# Example
recommend(
    notes=["Jasmine", "Rose", "Bergamot"],
    family="CITRUS",
    subfamily="FLORAL",
    gender="Male",
    k=5
)

In [ ]:
# --- Save the trained optimized model artifacts ---
import os
import joblib

os.makedirs("models", exist_ok=True)

opt_config = {
    "metric": str(best.metric),
    "k": int(best.n_neighbors),
    "weights": [float(x) for x in best_w],          # [w_ing, w_fam, w_sub, w_gen]
    "n_family": int(n_family),
    "n_subfamily": int(n_subfamily),
    "n_gender": int(n_gender),
    "hit_rate": float(best.hit_rate),
}

joblib.dump(df, "models/opt_perfume_df.pkl")
joblib.dump(mlb, "models/opt_mlb.pkl")
joblib.dump(ohe, "models/opt_ohe.pkl")
joblib.dump(opt_config, "models/opt_config.pkl")
np.save("models/opt_matrix.npy", build_matrix(best_w))

print("Saved optimized model -> models/opt_*.pkl + opt_matrix.npy")
print("Config:", opt_config)